
# MRI Together Hackathon: Optimising Diffusion MRI for Placental Imaging

## Can we get better MRI measurements using fewer acquisitions?

Diffusion MRI tells us about the placenta's microscopic structure and blood flow.

However, diffusion MRI scans take time. We cannot measure everything, so we need to decide **which measurements will give us the most useful information**.

In this hackathon, you will use simulations and machine learning to optimise a diffusion MRI acquisition protocol for the placenta.

You will:

1. **Simulate MRI signals** using a mathematical model of diffusion and blood flow in the placenta.
2. **Fit the model** to estimate tissue properties from noisy MRI measurements.
3. **Compare acquisition protocols** to see how the choice and number of measurements affect accuracy.
4. **Use machine learning to optimise a protocol**, selecting the most informative measurements.
5. **Evaluate the results** and experiment with ways to improve them.

**The challenge: Can you design an MRI acquisition protocol that accurately measures tissue properties using as few measurements as possible?**

In [ ]:
#import the relevant packages
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt


## 1. Simulating IVIM data

First, we will simulate some simple diffusion MRI data. To do this, we need two things:

1. **A model** describing how the MRI signal depends on tissue properties.
2. **An acquisition protocol** specifying which MRI measurements to collect.

For our model, we will use **Intravoxel Incoherent Motion (IVIM)**, which describes the MRI signal as a combination of tissue diffusion and blood flow.

The IVIM signal is given by:

$$
S(b) = f e^{-bD^*} + (1-f)e^{-bD}
$$

The model has three parameters:

- **Perfusion fraction ($f$):** The fraction of the signal associated with blood flowing through small vessels.
- **Pseudo-diffusivity ($D^*$):** The apparent diffusion coefficient associated with blood flow.
- **Diffusivity ($D$):** The diffusion coefficient of water within tissue.

The **b-value ($b$)** controls how sensitive the MRI measurement is to diffusion. By acquiring measurements at different b-values, we can estimate the three IVIM parameters.




In [ ]:
#let's define a function for simulating IVIM data
# note that this can take arrays of parameter values as input

def ivim(b, params, snr=None):
    b = np.asarray(b)
    params = np.asarray(params)

    f, d_p, d = np.moveaxis(params, -1, 0)

    signal = (
        f[..., None] * np.exp(-b * d_p[..., None])
        + (1 - f[..., None]) * np.exp(-b * d[..., None])
    )

    if snr is not None:
        sigma = 1 / snr
        signal = np.sqrt(
            (signal + np.random.normal(0, sigma, signal.shape))**2
            + np.random.normal(0, sigma, signal.shape)**2
        )

    return signal

Next, we need to define an **acquisition protocol**, which specifies the b-values at which we will measure the MRI signal.

We will use a protocol that we previously optimised using the **Cramér–Rao Lower Bound (CRLB)**, a mathematical approach for selecting measurements that minimise the theoretical uncertainty in parameter estimates.

For more details, see our [previous work](https://doi.org/10.1007/978-3-030-05831-9_20).


In [ ]:
# define baseline protocol from https://doi.org/10.1007/978-3-030-05831-9_20

# b-values in s/mm2
b_crlb = np.array([0, 5, 10, 20, 30, 50, 100, 200, 400, 600, 800, 1200, 1600])


In [ ]:
# We can now simulate and plot some IVIM data

f = 0.5 #perfusion fraction
D_p = 0.03 #pseudo-diffusivity
D = 0.003 #diffusivity
SNR = 30 #signal to noise ratio

s = ivim(b_crlb, [f, D_p, D], 30)

plt.plot(b_crlb, s, 'o')


## 2. Estimating tissue properties from MRI signals

We'll now estimate the underlying IVIM parameters from the diffusion MRI signal.

We will use nonlinear least-squares fitting to find the IVIM parameters that best reproduce each simulated signal.

To evaluate the fitting, we can compare the estimated parameters against the true parameters used in the simulations.

In [ ]:
# define a function for fitting the ivim model to data

from scipy.optimize import least_squares

from scipy.optimize import least_squares
import numpy as np

from tqdm.auto import tqdm


def fit_ivim(b, signals):
    perfusion_cutoff = 0.003
    signals = np.atleast_2d(signals)

    def fit_voxel(signal):
        result = least_squares(
            lambda params: ivim(b, params) - signal,
            x0=[0.2, 0.02, 0.001],
            bounds=([0, perfusion_cutoff, 0], [1, 0.1, perfusion_cutoff])
        )
        return result.x

    return np.array([fit_voxel(signal) for signal in tqdm(signals)])


In [ ]:
# now we simulate data for a range of biologically reasonable values
f_lim = [0, 0.8]
D_p_lim = [0.003, 0.1]
D_lim = [0, 0.003]

n_samples = 1000
snr = 30

params = np.column_stack([
    np.random.uniform(*f_lim, n_samples),
    np.random.uniform(*D_p_lim, n_samples),
    np.random.uniform(*D_lim, n_samples)
])

signals_crlb = ivim(b_crlb, params, snr=snr)



In [ ]:
# let's fit the model to the data

from tqdm.auto import tqdm

params_crlb_fit = fit_ivim(b_crlb,signals_crlb)

In [ ]:
# and finally plot the fitted parameters against the true parameters

param_names = ["Perfusion fraction", "Pseudo-diffusivity", "Diffusivity"]
param_units = ["", "mm²/s", "mm²/s"]

from scipy.stats import pearsonr

def plot_fits(params, params_fit):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for i, ax in enumerate(axes):
        true = params[:, i]
        fitted = params_fit[:, i]

        r, _ = pearsonr(true, fitted)

        ax.plot(true, fitted, 'o', markersize=2)
        ax.plot([true.min(), true.max()],
                [true.min(), true.max()], 'k--')

        ax.set_xlabel(f"True {param_names[i]} ({param_units[i]})")
        ax.set_ylabel(f"Fitted {param_names[i]} ({param_units[i]})")
        ax.set_title(f"R = {r:.3f}")

    plt.tight_layout()
    plt.show()

plot_fits(params, params_crlb_fit)


## 3. What if we could collect more measurements?

Our first protocol contains 13 b-values.

We will now create a much larger acquisition protocol containing 4 times more measurements **(52 b-values)**, which we call the *superdesign*.

The superdesign samples the diffusion signal more densely, giving us a larger set of candidate measurements from which to select.

We will simulate and fit IVIM signals using this protocol, then compare the results against the baseline.

The superdesign will also enable us to design a protocol with machine learning later.


In [ ]:
# define our superdesign
b_super = np.concatenate(([0], np.geomspace(1, 1600, 51)))

# simulate data using the superdesign
signals_super = ivim(b_super, params, snr=snr)

# fit to the simulated data
params_super_fit = fit_ivim(b_super,signals_super)


In [ ]:
# now let's plot our superdesign and the CRLB protocol side by side
plot_fits(params, params_super_fit)
plot_fits(params, params_crlb_fit)



## 4. Optimising the protocol using machine learning

We will now use **TADRED**, a machine-learning approach for selecting informative measurements from a larger acquisition protocol.

Rather than selecting b-values manually, TADRED learns which measurements are most useful for predicting the IVIM parameters.

Running the optimisation requires the following:

- **Inputs:** simulated MRI signals from the superdesign.
- **Targets:** the true IVIM parameters used to generate those signals.
- **Candidate measurements:** the 52 b-values in the superdesign.

TADRED jointly learns which measurements to retain and trains a neural network to estimate the IVIM parameters from those measurements.

We will run the optimisation using the **EDMRI** software package.

The optimisation produces:

- an acquisition protocol containing a subset of the original b-values, together with a trained neural network for parameter estimation.





In [ ]:
# now we'll apply tadred

# first we need to save the b-values, simulated input data, and simulated target data
np.savetxt("b_super.txt", b_super)
np.save("signals_super.npy", signals_super)
np.save("target_super.npy", params)


In [ ]:
# add some paths from the edmri packages
from edmri.paths import CONFIG_PATH

#choose the config file to use for the optimisation experiment
config_file = f"{CONFIG_PATH}/optimise_default.yaml"

#import the function
from edmri.optimise import optimise_experiment

In [ ]:
#run cli function to optimise the protocol and train the task network

!PYTHONUNBUFFERED=1 edmri-optimise \
    --force \
    --example IVIM \
    --config {CONFIG_PATH}/optimise_default.yaml \
    data.input=signals_super.npy \
    data.target=target_super.npy \
    data.superdesign=b_super.txt




In [ ]:
from edmri.paths import RESULTS_PATH

!ls {RESULTS_PATH}/optimisation/examples/IVIM/tadred_results





## 6. Testing the optimised protocol

Now that we have an optimised protocol, we need to evaluate how well it works.

We will:

1. Load the b-values selected by TADRED.
2. Simulate noisy IVIM signals using those b-values.
3. Apply the trained neural network to estimate the IVIM parameters.
4. Compare the estimated parameters with their true values.

We will compare three approaches:

| Protocol | Parameter estimation |
|---|---|
| CRLB baseline | Nonlinear least squares |
| Full superdesign | Nonlinear least squares |
| TADRED-optimised | Trained neural network |

**Important:** TADRED changes both the acquisition protocol and the fitting method. Therefore, any improvement could result from better measurement selection, better parameter estimation, or both.

**Question:** How could we design a fair comparison that separates these two effects?


In [ ]:
# let's find where the results are stored automatically
from edmri.paths import RESULTS_PATH
from pathlib import Path

# the results will be saved here
results_dir = Path(RESULTS_PATH, "optimisation/examples/IVIM/tadred_results")

# get the separate results
task_network_path = Path(RESULTS_PATH, "optimisation/examples/IVIM/tadred_results/trained_task_network.pt")
optimised_protocol_path = Path(RESULTS_PATH, "optimisation/examples/IVIM/tadred_results/optimised_protocol.txt")
optimised_indices_path = Path(RESULTS_PATH, "optimisation/examples/IVIM/tadred_results/optimised_indices.txt")


In [ ]:
# now we need to simulate some data with the tadred optimised protocol

b_tadred = np.loadtxt(optimised_protocol_path)

signals_tadred = ivim(b_tadred, params, snr=snr)

np.save("signals_tadred.npy", signals_tadred)


In [ ]:
# now we apply the tadred task network to the simulated data

result_apply = subprocess.run([
    "edmri-apply",
    "--force",
    "--example", "IVIM",
    "--config", str(CONFIG_PATH / "apply_default.yaml"),
    "data.input=signals_tadred.npy",
    f"task_network={task_network_path}"
], capture_output=True, text=True)

print(result_apply.stdout)
print(result_apply.stderr)

In [ ]:
# load the parameters estimated by the task network
params_tadred_fit = np.load(Path(RESULTS_PATH, "application/examples/IVIM/predictions.npy"))

In [ ]:
# let's compare b-values across protocols by plotting all the protocols together
plt.plot(b_super,signals_super[0,:],'.', label="superdesign")
plt.plot(b_crlb, signals_crlb[0,:], 'o', label="CRLB")
plt.plot(b_tadred, signals_tadred[0,:], 'x', label="TADRED")

plt.xlabel('b-value (s/mm2)')
plt.ylabel('signal')

plt.legend()

In [ ]:
# finally, let's compare the parameter estimates across all three protocols

plot_fits(params, params_tadred_fit)
plot_fits(params, params_crlb_fit)
plot_fits(params, params_super_fit)


## Additional tasks

Now that you've completed the basic workflow, try improving the results or extending the approach. Here are some ideas:

1. **Increase the training data:** We used a relatively small number of simulated voxels to train TADRED. Does increasing the number of training samples improve its performance?

2. **Improve the evaluation metrics:** So far, we've only used correlation coefficients to assess parameter estimation. Can you identify and implement more appropriate metrics, such as mean absolute error (MAE), root mean squared error (RMSE), or bias?

3. **Compare fitting methods fairly:** TADRED has an advantage because it uses a neural network for parameter estimation, whereas our other protocols use nonlinear least squares (LSQ). Try fitting the TADRED-optimised protocol using LSQ. How much of the improvement comes from the protocol rather than the fitting method?

4. **Explore the parameter ranges:** Does changing the minimum and maximum IVIM parameter values used in the simulations affect the optimised protocol or fitting accuracy?

5. **Extend the IVIM model:** Our current IVIM model assumes isotropic diffusion (the same in every direction). Can you extend it to include anisotropic diffusion? You will also need to modify the superdesign to include gradient directions.

6. **Try machine-learning-based fitting:** We've used LSQ for all parameter estimation except the EDMRI task network. Can you train a separate neural network to estimate IVIM parameters for the baseline or superdesign protocols? How does it compare with LSQ and TADRED?

7. **Experiment with the optimisation settings:** We've used the default EDMRI optimisation configuration (`optimise_default.yaml`). Create a new configuration file and experiment with different settings, such as the number of selected measurements, training iterations, or neural network parameters. How do these choices affect the results?

**Overall challenge:** Can you improve parameter estimation accuracy while keeping the number of MRI measurements as small as possible?
